In [ ]:
import alc
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import cv2

### Ejercicio 1 - DLT ecuaciones formales

In [ ]:
#Obtenemos las rutas de los archivos para luego hacer la lectura de los mismos
detected_corners_path = os.path.join('calib','3d','detected_corners.txt')
p_W_corners_path = os.path.join('calib','3d','p_W_corners.txt')

corners = np.loadtxt(p_W_corners_path,delimiter=',')

#Definimos a las variables U, V y W que guardan la primer, segunda y tercer columna, respectivamente, del archivo de los puntos del mundo real. Esto quiere decir que U es un array que contiene únicamente la primer coordenada de todos los puntos del archivo p_W_corners.txt (análogamente sucede lo mismo con V y W pero con la segunda y tercer coordenada de los puntos).
U, V, W = corners[:,0], corners[:,1], corners[:,2]

#Definimos en una variable G el contenido del archivo detected_corners.txt
G = np.loadtxt(detected_corners_path)

#Definimos la matriz Q como un array de ceros que luego completaremos
Q = np.zeros((2*corners.shape[0],12))
i = 0 #Definimos a en base a que foto construiremos nuestra Q, en este caso será la primer foto.
for j in range(corners.shape[0]): #Iteramos por cada punto en el mundo real y construimos la matriz Q tal como indica la ecuación (6)
    u = G[i][2*j] #Representa la primer coordenada asociada a la j-esimo punto del mundo real en la i-esima foto
    v = G[i][2*j+1] #Representa la segunda coordenada asociada a la j-esimo punto del mundo real en la i-esima foto

    #Escribimos cada par de filas asociadas a cada coordenada del mundo real
    Q[2*j, :] =     [U[j], V[j], W[j], 1,   0,    0,    0,   0, -u*U[j], -u*V[j], -u*W[j], -u]
    Q[2*j + 1, :] = [ 0,    0,    0,   0,  U[j], V[j], W[j], 1, -v*U[j], -v*V[j], -v*W[j], -v]

La matriz Q de la imagen 1 es:

In [ ]:
print(Q)

Sus valores singulares son:

In [ ]:
matrizU, valoresSingulares, matrizV = alc.svd_reducida(Q) 
print(valoresSingulares)

El menor valor singular se corresponde al ubicado en la última posición de la diagonal de de la matriz Sigma de una descomposición SVD, o sea el ubicado en la posición $(\min\{m,n\}, \min\{m,n\})$ (siendo $m$ y $n$ las dimensiones de la matriz).

En el caso del output de nuestra función, que no devuelve los valores singulares en una matriz sino en un array, se corresponde a el último valor del array valoresSingulares (puesto que estan ordenados de manera decreciente), el cual es:

In [ ]:
print(valoresSingulares[-1])

Dado que el vector $m$ es el asociado al valor singular más pequeño, entonces el mismo se corresponde a la última columna de la matriz $V$ (la demostración de lo dicho se encuentra en la celda de texto siguiente). Por lo que la norma $\|Qm\|$ es igual a:

In [ ]:
m = matrizV[:,-1]
Qm = alc.arroba(Q,m)
print(alc.norma(Qm,2))

Siendo $Q\in \mathbb{R}^{2n\times 12}$, ¿qué rango debe tener para asegurar una solución no nula única a menos de un factor de escala? ¿Cuántos puntos, como mínimo, aseguran la solución y qué condición deben cumplir?

Encontrar el vector $\mathbf{m}$ que resuelve la minimización de la ecuación 6.

<u>**Primer pregunta**</u>

Para responderla recurrimos al siguiente teorema:

* **Teorema de Rango-Nulidad** $\longrightarrow$ Sean $V$ y $W$ espacios vectoriales con $dim(V) < +\infty$ y sea $T:V \longrightarrow W$ una transformación lineal entonces $dim(Im(T)) + dim(Nu(T)) = dim(V)$

Considerando que debemos asegurar que hay una solución no nula única a menos de un factor de escala (esto último no interfiere en la consideración del teorema y respuesta del enunciado ya que el conjunto de soluciones es un subespacio y por lo tanto considera a todos sus múltiplos, con hallar una es suficiente y mantiene la propiedad de ser única), sabemos entonces que la $dim(Nu(\mathbf{Q})) = 1$, luego por el teorema podemos despejar $dim(Im(\mathbf{Q}))$, que es quien responde a la pregunta del rango. Por lo mencionado, podemos decir que $dim(Im(\mathbf{Q})) = dim(V) - dim(Nu(\mathbf{Q})) = dim(V) - 1 = 12 - 1 = 11$. El rango que debe tener $\mathbf{Q}$ para asegurar una solución no nula única a menos de un factor de escala es 11.

¿Quien es $V$? ¿Por qué $dim(V) = 12$? Podemos definir a $T$, la transformación lineal que representa nuestra matriz $\mathbf{Q}$, tal que (por las dimensiones de $\mathbf{Q}$) vemos que toma elementos de $R^{12}$ y los lleva a $R^{2n}$, siendo entonces $T:R^{12} \longrightarrow R^{2n}$. Por ello, el espacio de salida es $V = R^{12}$ y $dim(R^{12}) = 12$ $\Longrightarrow$ $dim(V) = 12$.

<u>**Segunda pregunta**</u>

¿Cuántos puntos, como mínimo, aseguran la solución? Prestando atención a esta parte de la pregunta vemos que la misma también se traduce a cuantas filas necesitamos como mínimo para que la respuesta de la anterior pregunta sea válida (o sea, $dim(Im(\mathbf{Q})) = 11$ y, por lo tanto, $dim(Nu(\mathbf{Q})) = 1$) ¿Por qué filas si nos preguntan puntos? Porque cada par de filas en la matriz $\mathbf{Q}$ representa un punto, y además sabemos que el rango de una matriz (que es igual a la dimensión de la imagen de la matriz) es mayor o igual a uno y menor o igual a la cantidad de filas de ella. Por lo que podemos concluir que $dim(Im(\mathbf{Q})) = 11 \leq 2n$, más específicamente: $\frac{11}{2} = 5.5 \leq n$. Entonces (considerando que $n$ es un número natural) la cantidad de puntos mínimos que aseguran la solución son $6$. 

¿Y qué condición deben cumplir? Como $dim(Im(\mathbf{Q})) = 11$ entonces sabemos que las filas deben de ser linealmente independientes entre sí, ya que le hallamos un techo al decir que $n = 6$ por lo menos, pero también debemos garantizar su piso para que la dimensión del núcleo no crezca (ya que de esa manera perderiamos la condición $dim(Nu(\mathbf{Q})) = 1$ y sería mayor a uno). Para ello, la matriz $\mathbf{Q}$ es una matriz que tiene 11 filas necesariamente linealmente independientes. Traduciendo esa información a los puntos $U,V,W$ logramos entender que será necesario que hayan 6 puntos no coplanares, o sea que hayan 6 puntos distribuidos de tal manera que no exista un plano que pueda contenerlos a la vez.

<u>**Hallar el vector**</u> $\mathbf{m}$

Sea $Q$ una matriz de $\mathbb{R}^{m \times n}$ tal que, por su descomposición SVD, puedo decir que $Q = U \Sigma V^t$, con $U \in \mathbb{R}^{m \times m}$, $\Sigma \in \mathbb{R}^{m \times n}$ y $V \in \mathbb{R}^{n \times n}$. Siendo $U$ y $V$ ortogonales.

Donde la matriz $V$ son los autovectores asociados a los autovalores de la matriz $Q^t Q$ escritos a modo de columnas, en orden asociado a los valores singulares dados en la matriz $\Sigma$ y forman una base ortonormal; la matriz $\Sigma$ es una matriz que contiene en la diagonal dada desde la posición $(1,1)$ hasta $(\min\{m,n\}, \min\{m,n\})$ los valores singulares de $Q$ de forma decreciente (o sea, $\sigma_i = \sqrt{\lambda_i}$, siendo $\lambda_i$ los autovalores de la matriz $Q^t Q$), y en el resto de sus posiciones hay solamente ceros; y la matriz $U$ está dada por los vectores $u_i  = \frac{Q v_i}{\sigma_i}$ a modo de columna siendo dicha igualdad para cuando $\sigma_i \neq 0$, luego se obtienen el resto de vectores (en caso de haber valores singulares nulos o que $m > n$) extendiendo a una base ortonormal.

Dado que por enunciado queremos hallar el vector $\mathbf{m}$ tal que minimice la siguiente expresión

$$\min\limits_{\mathbf{m}} \|\mathbf{Qm}\|^2$$

Podemos decir que, por la descomposición SVD explicada al inicio del texto, es igual a hallar el $\mathbf{m}$ que minimice la expresión dada con la matriz $Q$ escrita como multiplicación de $U$, $\Sigma$ y $V^t$.

$$\min\limits_{\mathbf{m}} \|\mathbf{Qm}\|^2 = \min\limits_{\mathbf{m}} \|U \Sigma V^t m\|^2$$

Dado que $U$ y $V^t$ son ortogonales, pues si $V$ es ortogonal entonces $V^t$ es ortogonal también, sabemos que $\|\mathbf{Ux}\| = \|\mathbf{x}\|$ $\forall x \in \mathbb{R^m}$ y $\|\mathbf{V^tx}\| = \|\mathbf{x}\| \, \forall x \in \mathbb{R^n}$, por lo que podemos reescribir la anterior expresión de la siguiente manera

$$\min\limits_{\mathbf{m}} \|\Sigma V^t m\|^2$$

Esta paso es valido pues la multiplicación $\Sigma V^t m$ devuelve un vector, y como $U$ es ortogonal podemos garantizar que $\| U \Sigma V^t m\| = \| U (\Sigma V^t m)\| = \|\Sigma V^t m\|$

Para resolver la minimización podemos decir que $\Sigma V^t m$ = $\Sigma (V^t m)$ = $\Sigma y$ con $y$ vector tal que $y = V^t m$. Esta reescritura nos es útil y la elegimos ya que podemos concluir que $\|y\| = \|V^t m\| = \|m\| = 1$ pues $V^t$ es ortogonal y, por definición de la ecuación $(7)$, $\|m\| = 1$. Entonces lo anterior es igual a la siguiente expresión

$$\min\limits_{\mathbf{y}} \|\Sigma y\|^2$$


Lo que nos interesa ahora es hallar un vector unitario $y$ tal que el mismo minimiza la expresión. Desarrollamos la norma al cuadrado de la minimización y vemos que

$$\|\Sigma y\|^2 = \displaystyle\sum_{i=1}^{n} (\sigma_i y_i)^2 = \displaystyle\sum_{i=1}^{n} \sigma_i^2 y_i^2$$

Dado que los valores singulares se encuentran ordenados de forma decreciente (o sea, $\sigma_1 \ge \sigma_2 \ge ... \ge \sigma_n \ge 0$) entonces podemos garantizar que $\sigma_i \ge \sigma_n \, \forall i = 1,...,n$. Valiendo también $\sigma_i^2 \ge \sigma_n^2 \, \forall i = 1,...,n$. Teniendo dicha desigualdad podemos acotar la expresión desarrollada de la siguiente forma

$$\|\Sigma y\|^2 = \displaystyle\sum_{i=1}^{n} \sigma_i^2 y_i^2 \ge \displaystyle\sum_{i=1}^{n} \sigma_n^2 y_i^2 = \sigma_n^2 \displaystyle\sum_{i=1}^{n} y_i^2 = \sigma_n^2 \|y\|^2 = \sigma_n^2 * 1 = \sigma_n^2$$

(La suma se realiza hasta $n$ y no $m$ pues en el caso de nuestra $Q$ tenemos que $m \ge n$)

De esta manera logramos acotar inferiormente la minimización y podemos indicar que la misma se obtiene cuando $y = e_n$, puesto que $\|e_n\| = 1$ y $\Sigma e_n = (0,...,0,\sigma_n,0,...,0)^t$. Entonces

$$\|\Sigma e_n\|^2 = \|(0,...,0,\sigma_n,0,...,0)^t\|^2 = \sigma_n^2$$

Reacomodando la reescritura que dimos para facilitar la demostración vemos que $y = e_n = V^t m$. Puesto que $V^t$ es ortogonal, vemos que  $Ve_n = m$ y $Ve_n = v_n = m$ (siendo $v_n$ la $n$-ésima columna de $V$, y como $V$ es ortogonal y sus columnas forman una $BON$ entonces $\|v_n\| = 1$).

$\therefore$ El vector $m$ que minimiza la expresión $\|\mathbf{Qm}\|^2$ es $m = v_n$.